# Từ LeNet tới ResNet và ViT trên CIFAR-10

CIFAR-10: 60 000 ảnh màu 32×32 (50 000 train, 10 000 test), 10 lớp cân bằng. Bạn dựng ba thế hệ kiến trúc
bằng PyTorch (dùng `nn.Conv2d` thoải mái, vì conv bằng numpy bạn đã tự viết ở bài Basic 01): **LeNet-5**
(1998), **ResNet** cho CIFAR (2015, residual + BatchNorm), và một **Vision Transformer** nhỏ với patch
embedding và multi-head self-attention tự viết. Rồi train cả ba cùng một vòng train, cùng ngân sách, và
đọc kết quả: tham số, thời gian, accuracy, tác dụng của augmentation và của đường tắt residual.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải CIFAR-10 vào `data/raw/cifar10` nếu chưa có (137 MB).
- Mặc định mọi thí nghiệm chạy trên CPU với 10 000 ảnh train (vài phút mỗi model). Có GPU thì đặt
  `DEVICE = "cuda"` (kernel `ai-gpu` của `setup/gpu`) và tăng `N_TRAIN`, `EPOCHS` ở cell thí nghiệm.

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải CIFAR-10 nếu chưa có (137 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy/torch
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

CIFAR = ensure("cifar10")
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import io
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn

CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

## 1. Đọc dữ liệu, chuẩn hoá theo kênh

Bản mirror Hugging Face lưu CIFAR-10 thành hai tệp parquet trong `plain_text/`:
`train-00000-of-00001.parquet` và `test-00000-of-00001.parquet`, cột `img` là dict `{"bytes": <PNG>}`,
cột `label` là số 0–9 theo thứ tự `CLASSES`.

- `load_cifar10(root, split, limit=None)`: `split` là `"train"` hoặc `"test"`. Trả `(X, y)`: `X` mảng `uint8`
  shape `(N, 32, 32, 3)` (RGB, thứ tự dòng như trong tệp), `y` mảng `int64` shape `(N,)`. `limit` là số ảnh
  đầu cần giải mã (`None`: tất cả); giải mã 50 000 PNG mất khoảng 15 giây.
- `channel_stats(X)`: mean và std **theo từng kênh** R, G, B của ảnh đã chia 255, tính trên mọi pixel mọi
  ảnh. Trả hai mảng shape `(3,)`.
- `to_tensor(X, mean, std)`: `uint8 (N, H, W, 3)` → `torch.float32 (N, 3, H, W)`, chia 255 rồi chuẩn hoá
  $$(x - \mu_c)/\sigma_c$$ theo kênh. Dùng **thống kê của tập train** cho cả train và test.

In [ ]:
def load_cifar10(root: Path, split: str, limit: int | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Đọc plain_text/{split}-00000-of-00001.parquet → (X uint8 (N, 32, 32, 3) RGB, y int64 (N,)).

    `limit`: chỉ lấy `limit` ảnh đầu (None: tất cả).
    """
    df = pd.read_parquet(Path(root) / "plain_text" / f"{split}-00000-of-00001.parquet")
    if limit is not None:
        df = df.iloc[:limit]
    X = np.stack([np.asarray(Image.open(io.BytesIO(cell["bytes"])).convert("RGB")) for cell in df["img"]])
    return X, df["label"].to_numpy(dtype=np.int64, copy=True)


def channel_stats(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Mean và std theo kênh của X / 255, X shape (N, H, W, 3). Trả (mean (3,), std (3,))."""
    flat = X.reshape(-1, 3).astype(np.float64) / 255
    return flat.mean(axis=0), flat.std(axis=0)


def to_tensor(X: np.ndarray, mean, std) -> torch.Tensor:
    """uint8 (N, H, W, 3) → float32 (N, 3, H, W), chia 255 rồi chuẩn hoá theo kênh."""
    x = torch.from_numpy(np.ascontiguousarray(X)).permute(0, 3, 1, 2).float() / 255
    m = torch.as_tensor(np.asarray(mean), dtype=torch.float32).view(1, 3, 1, 1)
    s = torch.as_tensor(np.asarray(std), dtype=torch.float32).view(1, 3, 1, 1)
    return (x - m) / s

In [ ]:
# explore: đọc dữ liệu thật, thống kê, xem vài ảnh
import matplotlib.pyplot as plt

t0 = time.perf_counter()
X_train, y_train = load_cifar10(CIFAR, "train")
X_test, y_test = load_cifar10(CIFAR, "test")
print(f"đọc xong trong {time.perf_counter() - t0:.1f} s:", X_train.shape, X_train.dtype, X_test.shape, y_train[:8].tolist())
print("số ảnh mỗi lớp (train):", np.bincount(y_train).tolist())
MEAN, STD = channel_stats(X_train)
print("mean theo kênh:", MEAN.round(4), "| std theo kênh:", STD.round(4))
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, label in zip(axes.flat, X_train[:16], y_train[:16]):
    ax.imshow(img)
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis("off")
plt.show()

## 2. Augmentation trên batch: random crop + horizontal flip

Augmentation chuẩn của CIFAR từ bài báo ResNet: đệm 4 pixel mỗi phía rồi cắt ngẫu nhiên lại 32×32 (ảnh dịch
tối đa 4 pixel mỗi chiều), và lật ngang với xác suất 0.5. Áp **trên tensor đã chuẩn hoá**, chỉ khi train.

`random_crop_flip(x, generator, pad=4)`: `x` shape `(N, C, H, W)`. Mỗi ảnh lấy độ dịch riêng
`(dy, dx)` trong `0..2*pad` và cờ lật riêng; vùng đệm là số 0 (ở không gian đã chuẩn hoá, 0 là màu trung
bình). Mọi số ngẫu nhiên lấy từ `generator` (`torch.Generator`), để cùng seed ra cùng kết quả. Trả tensor
cùng shape.

In [ ]:
def random_crop_flip(x: torch.Tensor, generator: torch.Generator, pad: int = 4) -> torch.Tensor:
    """Random crop (đệm `pad` số 0 mỗi phía, cắt lại H x W) + lật ngang p = 0.5, riêng cho từng ảnh."""
    N, _, H, W = x.shape
    padded = F.pad(x, (pad, pad, pad, pad))
    oy = torch.randint(0, 2 * pad + 1, (N,), generator=generator).tolist()
    ox = torch.randint(0, 2 * pad + 1, (N,), generator=generator).tolist()
    flip = (torch.rand(N, generator=generator) < 0.5).tolist()
    out = torch.empty_like(x)
    for i in range(N):
        crop = padded[i, :, oy[i]:oy[i] + H, ox[i]:ox[i] + W]
        out[i] = crop.flip(-1) if flip[i] else crop
    return out

## 3. LeNet-5 và ResNet cho CIFAR

**LeNet-5** (LeCun 1998), bản hiện đại hoá cho ảnh màu 32×32: `Conv(3→6, 5) → ReLU → MaxPool 2 →
Conv(6→16, 5) → ReLU → MaxPool 2 → Flatten (16·5·5 = 400) → Linear 120 → ReLU → Linear 84 → ReLU → Linear 10`.

**`BasicBlock(in_ch, out_ch, stride=1, residual=True)`** của ResNet:

$$
y = \mathrm{ReLU}\big(\mathrm{BN}_2(\mathrm{conv}_2(\mathrm{ReLU}(\mathrm{BN}_1(\mathrm{conv}_1 x)))) + \mathrm{shortcut}(x)\big)
$$

- `conv1`: 3×3, `stride`, padding 1; `conv2`: 3×3, stride 1, padding 1; cả hai `bias=False` (BN ngay sau đã
  có shift). Đặt tên thuộc tính `conv1, bn1, conv2, bn2, shortcut`.
- `shortcut`: identity (`nn.Identity()`, không tham số) khi `stride == 1` và `in_ch == out_ch`; ngược lại
  conv 1×1 với `stride`, `bias=False`, rồi BN (projection, "option B" của bài báo).
- `residual=False` bỏ hẳn phép cộng shortcut: đó là mạng "plain" để so.
- ReLU cuối nằm **sau** phép cộng.

**`ResNetCifar(depth=20, num_classes=10, residual=True)`** (He et al. 2015, mục 4.2): `depth = 6n + 2`.
Stem `Conv 3×3 (3→16), BN, ReLU`; ba stage, mỗi stage `n` block với 16, 32, 64 kênh; block đầu của stage 2
và 3 có `stride=2`; rồi **global average pooling** và `Linear(64 → num_classes)`. Không có tầng Linear
lớn nào sau flatten, nên model nhận ảnh cỡ bất kỳ.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16 * 5 * 5, 120), nn.ReLU(), nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, residual: bool = True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)
        return F.relu(out)


class ResNetCifar(nn.Module):
    def __init__(self, depth: int = 20, num_classes: int = 10, residual: bool = True):
        super().__init__()
        assert (depth - 2) % 6 == 0, "depth = 6n + 2"
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        blocks, in_ch = [], 16
        for stage, ch in enumerate([16, 32, 64]):
            for i in range(n):
                blocks.append(BasicBlock(in_ch, ch, 2 if stage > 0 and i == 0 else 1, residual))
                in_ch = ch
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.blocks(self.stem(x)).mean(dim=(2, 3)))

## 4. Vision Transformer nhỏ

ViT (Dosovitskiy et al. 2020) coi ảnh là một chuỗi patch:

1. **`PatchEmbed(patch, in_ch, dim)`**: cắt ảnh thành các patch `patch × patch` không chồng nhau, chiếu mỗi
   patch (vector `in_ch·patch²` số) thành vector `dim` chiều bằng một phép tuyến tính dùng chung. Một
   `nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)` làm đúng việc đó; đặt tên thuộc tính `proj`.
   Output shape `(B, số patch, dim)`, patch xếp theo hàng rồi cột.
2. **`MultiHeadSelfAttention(dim, heads)`**: tự viết, không dùng `nn.MultiheadAttention` hay
   `F.scaled_dot_product_attention`. Thuộc tính `qkv = nn.Linear(dim, 3*dim)` (output xếp `[q | k | v]`,
   mỗi phần chia `heads` head liền nhau, mỗi head `dim/heads` chiều) và `proj = nn.Linear(dim, dim)`:

$$
\mathrm{head}_i = \mathrm{softmax}\!\left(\frac{Q_iK_i^\top}{\sqrt{d_h}}\right)V_i, \qquad
\mathrm{out} = \mathrm{concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\,W_{proj} + b_{proj}
$$

3. **`ViT(img_size=32, patch=4, dim=128, depth=6, heads=4, mlp_ratio=2, num_classes=10)`**: patch embedding,
   ghép **CLS token** học được vào đầu chuỗi (`cls_token` shape `(1, 1, dim)`), cộng **position embedding**
   học được (`pos_embed` shape `(1, số patch + 1, dim)`, khởi tạo phân phối chuẩn std 0.02), `depth` block
   **pre-norm** (`x = x + attn(LN(x))`, `x = x + MLP(LN(x))`, MLP là `Linear(dim, mlp_ratio·dim) → GELU →
   Linear(mlp_ratio·dim, dim)`), LayerNorm cuối, rồi `Linear(dim, num_classes)` trên vector của CLS token.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, patch: int = 4, in_ch: int = 3, dim: int = 128):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, C, H, W) → (B, (H/patch)·(W/patch), dim)."""
        return self.proj(x).flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim: int, heads: int):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (B, N, dim) → (B, N, dim)."""
        B, N, D = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(D // self.heads)       # (B, h, N, N)
        out = att.softmax(dim=-1) @ v                                          # (B, h, N, d_h)
        return self.proj(out.transpose(1, 2).reshape(B, N, D))


class TransformerBlock(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_ratio: int = 2):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_ratio * dim), nn.GELU(), nn.Linear(mlp_ratio * dim, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class ViT(nn.Module):
    def __init__(self, img_size: int = 32, patch: int = 4, dim: int = 128, depth: int = 6, heads: int = 4,
                 mlp_ratio: int = 2, num_classes: int = 10):
        super().__init__()
        self.patch_embed = PatchEmbed(patch, 3, dim)
        n_patches = (img_size // patch) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[TransformerBlock(dim, heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(len(x), -1, -1), x], dim=1) + self.pos_embed
        return self.head(self.norm(self.blocks(x))[:, 0])

In [ ]:
# explore: số tham số và shape output của các model
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

for name, m in [("LeNet5", LeNet5()), ("Plain-20", ResNetCifar(20, residual=False)), ("ResNet-20", ResNetCifar(20)),
                ("ResNet-56", ResNetCifar(56)), ("ViT d128x6", ViT())]:
    print(f"{name:11s} {count_params(m):>9,} tham số | output {tuple(m(torch.zeros(2, 3, 32, 32)).shape)}")

## 5. Một vòng train cho mọi model

- `train(model, X, y, *, epochs, lr, batch_size=128, weight_decay=0.05, seed=0, augment=True, device="cpu")`:
  `X` là tensor đã chuẩn hoá `(N, 3, 32, 32)`, `y` tensor `int64`. `torch.manual_seed(seed)` ở đầu và một
  `torch.Generator` seed `seed` cho mọi số ngẫu nhiên của vòng train (xáo, augmentation). Mỗi epoch xáo bằng
  `torch.randperm(N, generator=g)`, chạy minibatch (augment bằng `random_crop_flip` nếu `augment`),
  `F.cross_entropy`, **AdamW** (`lr`, `weight_decay`) và **`OneCycleLR`** theo **bước** (`max_lr=lr`,
  `total_steps = epochs * ceil(N / batch_size)`, `pct_start=0.15`). Trả danh sách loss train trung bình
  **theo mẫu** của từng epoch. Kết thúc ở chế độ `eval()`.
- `evaluate(model, X, y, batch_size=500, device="cpu") -> float`: accuracy ở chế độ `eval()`, không tính
  gradient.

In [ ]:
def train(model: nn.Module, X: torch.Tensor, y: torch.Tensor, *, epochs: int, lr: float, batch_size: int = 128,
          weight_decay: float = 0.05, seed: int = 0, augment: bool = True, device: str = "cpu") -> list[float]:
    """AdamW + OneCycleLR theo bước. Trả loss train trung bình theo mẫu của từng epoch."""
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=epochs * math.ceil(len(X) / batch_size), pct_start=0.15)
    history = []
    for _ in range(epochs):
        perm = torch.randperm(len(X), generator=g)
        total = 0.0
        for start in range(0, len(X), batch_size):
            idx = perm[start:start + batch_size]
            xb = random_crop_flip(X[idx], g) if augment else X[idx]
            loss = F.cross_entropy(model(xb.to(device)), y[idx].to(device))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
        history.append(total / len(X))
    model.eval()
    return history


@torch.no_grad()
def evaluate(model: nn.Module, X: torch.Tensor, y: torch.Tensor, batch_size: int = 500, device: str = "cpu") -> float:
    model.to(device).eval()
    correct = 0
    for start in range(0, len(X), batch_size):
        pred = model(X[start:start + batch_size].to(device)).argmax(dim=1).cpu()
        correct += int((pred == y[start:start + batch_size]).sum())
    return correct / len(X)